In [1]:
import cords
import numpy as np
import torch
import torch
import torch.nn as nn
import torch.optim as optim
import logging
from dotmap import DotMap
from torchvision import datasets, transforms

from cords.utils.data.dataloader.SL.adaptive import GradMatchDataLoader
from cords.utils.models import ResNet18

/home/mila/a/ahmedm/.conda/envs/gradmatch/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── 1. Data ──────────────────────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

full_train = datasets.CIFAR10(root='data/', train=True,  download=False, transform=transform_train)
testset    = datasets.CIFAR10(root='data/', train=False, download=False, transform=transform_test)

# CORDS expects a validation split (10% by convention, matching the paper)
n_val   = int(0.1 * len(full_train))
n_train = len(full_train) - n_val
trainset, valset = torch.utils.data.random_split(full_train, [n_train, n_val],
                                                  generator=torch.Generator().manual_seed(42))

BATCH = 128
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH, shuffle=False, pin_memory=True)
valloader   = torch.utils.data.DataLoader(valset,   batch_size=BATCH, shuffle=False, pin_memory=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=BATCH, shuffle=False, pin_memory=True)

# ── 2. Model ──────────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = ResNet18(num_classes=10)
# CIFAR10 needs a smaller initial conv (no stride, no maxpool) — same tweak train_sl.py applies --> to check
model.conv1   = nn.Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
model.maxpool = nn.Identity()
model = model.to(device)

# ── 3. Loss ───────────────────────────────────────────────────────────────────
criterion      = nn.CrossEntropyLoss()
criterion_nored = nn.CrossEntropyLoss(reduction='none')   # ← GradMatch needs per-sample loss

# ── 4. Optimizer / Scheduler (paper: SGD lr=0.01, cosine annealing, 350 epochs) ──
LR        = 0.01
NUM_EPOCHS = 350 # to check
optimizer  = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4, nesterov=True)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# ── 5. GradMatch dss_args ─────────────────────────────────────────────────────
dss_args = DotMap({
    # --- required by GradMatchDataLoader ---
    'type'           : 'GradMatch',       # or 'GradMatchPB' (per-batch), 'GradMatch-Warm'
    'model'          : model,
    'loss'           : criterion_nored,
    'eta'            : LR,                # learning rate fed to the OMP solver
    'num_classes'    : 10,
    'num_epochs'     : NUM_EPOCHS,
    'device'         : device,
    'valid'          : False,
    # --- subset selection hyperparameters ---
    'fraction'       : 0.3,              # 0.1 / 0.2 / 0.3 tested in the paper
    'select_every'   : 20,               # re-select every R epochs
    'kappa'          : 0,                # 0 → no warm start; >0 → warm-start after kappa epochs
    'linear_layer'   : False,            # True = use only last-layer gradients (faster, slight accuracy drop)
    'selection_type' : 'PerClassPerGradient',  # paper default; 'PerBatch' for GradMatchPB
    'greedy'         : 'Stochastic',     # 'Stochastic' (paper default) or 'Naive'
    'collate_fn'     : None,
    'v1'             : True,
    'lam'            : 0.5,
    'eps'            :1e-100,
})

# ── 6. Build the GradMatch dataloader ─────────────────────────────────────────
logger   = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

dataloader = GradMatchDataLoader(
    trainloader, valloader, dss_args, logger,
    batch_size  = BATCH,
    shuffle     = True,
    pin_memory  = True,
)

In [3]:
# ── 7. Training loop ──────────────────────────────────────────────────────────
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    for inputs, targets, weights in dataloader:   # ← note the extra `weights`
        inputs, targets, weights = (inputs.to(device),
                                    targets.to(device),
                                    weights.to(device))
        optimizer.zero_grad()
        outputs = model(inputs)
        losses  = criterion_nored(outputs, targets)
        # weighted loss — CORDS convention
        loss    = torch.dot(losses, weights / weights.sum())
        loss.backward()
        optimizer.step()

    scheduler.step()

    # Optional: inspect which indices were selected this epoch
    # dataloader.selected_idxs  →  dict {epoch: [idx, ...]}

    if epoch % 50 == 0:
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in testloader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(1)
                correct += pred.eq(y).sum().item()
                total   += y.size(0)
        print(f"Epoch {epoch}  Test acc: {100.*correct/total:.2f}%")

INFO:__main__:Epoch: 21, GradMatch subset selection finished, takes 19.4636. 
INFO:__main__:Epoch: 41, GradMatch subset selection finished, takes 73.7664. 


Epoch 50  Test acc: 57.29%


INFO:__main__:Epoch: 61, GradMatch subset selection finished, takes 47.4292. 
INFO:__main__:Epoch: 81, GradMatch subset selection finished, takes 19.6083. 


Epoch 100  Test acc: 77.12%


INFO:__main__:Epoch: 101, GradMatch subset selection finished, takes 19.5707. 
INFO:__main__:Epoch: 121, GradMatch subset selection finished, takes 19.5991. 
INFO:__main__:Epoch: 141, GradMatch subset selection finished, takes 19.6296. 


...

Epoch 250  Test acc: 84.86%


INFO:__main__:Epoch: 261, GradMatch subset selection finished, takes 19.8115. 
INFO:__main__:Epoch: 281, GradMatch subset selection finished, takes 47.1177. 


Epoch 300  Test acc: 86.93%


INFO:__main__:Epoch: 301, GradMatch subset selection finished, takes 19.8786. 
INFO:__main__:Epoch: 321, GradMatch subset selection finished, takes 20.5760. 
INFO:__main__:Epoch: 341, GradMatch subset selection finished, takes 19.8066. 


Epoch 350  Test acc: 87.03%


In [3]:
idxes = dataloader.subset_indices
gammas = dataloader.subset_weights


np.save('gradmatch_indices_13500points.npy',np.array(idxes),allow_pickle=True)
np.save('gradmatch_weights_13500points.npy',np.array(gammas),allow_pickle=True)

In [4]:
import numpy as np
import torch

# Cast loaded indices to standard integers and gammas to float32
loaded_indices = np.load('gradmatch_indices_13500points.npy')
idxes = [int(i) for i in loaded_indices.ravel()]

loaded_weights = np.load('gradmatch_weights_13500points.npy')
gammas = torch.from_numpy(loaded_weights).float()

In [5]:
import torch
from torch.utils.data import Dataset, Subset

class WeightedSubsetDataset(Dataset):
    def __init__(self, original_dataset, indices, weights):
        """
        Wraps a dataset to yield (input, target, weight) triplets.
        """
        self.subset = Subset(original_dataset, indices)
        # Ensure weights is a 1D tensor of the same length as the indices
        self.weights = torch.tensor(weights).float() if not torch.is_tensor(weights) else weights.float()
        
        assert len(self.subset) == len(self.weights), "Mismatch between number of indices and weights."

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        w = self.weights[idx]
        return x, y, w

# Assuming `idxs` and `gammas` are already in memory from your previous cell
weighted_train_dataset = WeightedSubsetDataset(trainset, idxes, gammas)

weighted_trainloader = torch.utils.data.DataLoader(
    weighted_train_dataset, 
    batch_size=BATCH, 
    shuffle=True, 
    pin_memory=True
)

# ── 2. Initialize a Fresh Model for Retraining ────────────────────────────────
print(f"Initializing training on fixed subset of size: {len(weighted_train_dataset)}")

retrain_model_gradmatch = ResNet18(num_classes=10)
retrain_model_gradmatch.conv1 = nn.Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
retrain_model_gradmatch.maxpool = nn.Identity()
retrain_model_gradmatch = retrain_model_gradmatch.to(device)

retrain_optimizer = optim.SGD(retrain_model_gradmatch.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4, nesterov=True)
retrain_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(retrain_optimizer, T_max=NUM_EPOCHS)
criterion_nored = nn.CrossEntropyLoss(reduction='none')

# ── 3. Retraining Loop ────────────────────────────────────────────────────────
for epoch in range(1, NUM_EPOCHS + 1):
    retrain_model_gradmatch.train()
    
    for inputs, targets, weights in weighted_trainloader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        weights = weights.to(device)
        
        retrain_optimizer.zero_grad()
        outputs = retrain_model_gradmatch(inputs)
        
        # Calculate per-sample losses
        losses = criterion_nored(outputs, targets)
        
        # Apply weights: dot product of losses and normalized weights
        # Avoid division by zero if batch weights sum to 0
        weight_sum = weights.sum()
        if weight_sum > 0:
            loss = torch.dot(losses, weights / weight_sum)
        else:
            loss = (losses * weights).mean()
            
        loss.backward()
        retrain_optimizer.step()
        
    retrain_scheduler.step()
    
    # Validation step
    if epoch % 50 == 0 or epoch == NUM_EPOCHS:
        retrain_model_gradmatch.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in testloader:
                x, y = x.to(device), y.to(device)
                pred = retrain_model_gradmatch(x).argmax(1)
                correct += pred.eq(y).sum().item()
                total += y.size(0)
                
        print(f"Epoch {epoch:03d} | Retrain Test Acc: {100. * correct / total:.2f}%")

Initializing training on fixed subset of size: 13500
Epoch 050 | Retrain Test Acc: 83.81%
Epoch 100 | Retrain Test Acc: 85.74%
Epoch 150 | Retrain Test Acc: 87.65%
Epoch 200 | Retrain Test Acc: 87.91%
Epoch 250 | Retrain Test Acc: 88.02%
Epoch 300 | Retrain Test Acc: 87.89%
Epoch 350 | Retrain Test Acc: 87.96%
